# 03 · Exploratory data analysis (EDA)

Describe the cohort, the outcome distribution and the PASI trajectories **before** any modelling.

⚠️ If `SOURCE = "fictitious"`, the numbers are meaningless: this only tests the code.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import params, path
pd.set_option("display.max_columns", 50)

In [ ]:
# Data source: "fictitious" (test data) or "kruczek_il17" (Polish public dataset)
SOURCE = "fictitious"

if SOURCE == "fictitious":
    from src.data.loaders.fictitious import load
elif SOURCE == "kruczek_il17":
    from src.data.loaders.kruczek_il17 import load

from src.data.clean import clean
raw = load()
tables, cleaning_log = clean(raw)
{k: len(v) for k, v in tables.items()}

## 1. Model 2 cohort and outcomes

In [ ]:
from src.data.cohorts import model2_cohort
from src.outcomes.response import compute_response
from src.features.baseline import model2_features

c2 = model2_cohort(tables)
resp = compute_response(tables, c2)
feat = model2_features(tables, c2)
df2 = c2.merge(resp, on=["patient_id", "episode_id"]).merge(
    feat.drop(columns="drug_class"), on=["patient_id", "episode_id"])
print(f"{len(df2)} treatment episodes from {df2.patient_id.nunique()} patients")
resp.status.value_counts()

**Why are some episodes not assessable?** This table goes into the thesis (flow of the cohort).

## 2. Table 1: baseline characteristics by treatment class

In [ ]:
num = ["age", "bmi", "disease_duration", "pasi_baseline", "bsa_baseline", "dlqi_baseline"]
binv = ["sex_male", "psoriatic_arthritis", "difficult_sites", "prev_failure_inefficacy"]

def table1(d, by):
    g = d.groupby(by)
    t = pd.DataFrame({"n": g.size()})
    for v in num:
        t[v] = g[v].median().round(1).astype(str) + " (" + g[v].quantile(.25).round(1).astype(str) + "–" + g[v].quantile(.75).round(1).astype(str) + ")"
    for v in binv:
        t[v + " %"] = (100 * g[v].mean()).round(0)
    return t.T

t1 = table1(df2, "drug_class")
t1.to_csv(path("tables") / "table1_model2.csv")
t1

## 3. Outcome distribution
The key question: are there enough **non-responders** to train a model? Rule of thumb: at least ~10 events of the less frequent class per predictor.

In [ ]:
assessed = df2.dropna(subset=["pasi90"])
out = assessed.groupby("drug_class")[["pasi75", "pasi90", "dlqi_improved"]].agg(["mean", "count"]).round(2)
print("Overall:", assessed[["pasi75", "pasi90", "dlqi_improved"]].mean().round(2).to_dict())
print("Non-responders (PASI90 = 0):", int((assessed.pasi90 == 0).sum()))
out

In [ ]:
ax = assessed.groupby("drug_class")[["pasi75", "pasi90"]].mean().mul(100).plot.bar(figsize=(8, 4))
ax.set_ylabel("% of patients"); ax.set_xlabel(""); ax.set_title("Response at the class-specific window")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(path("figures") / "response_by_class.png", dpi=200); plt.show()

## 4. PASI trajectories
One line per treatment episode, time since treatment start. This is the longitudinal view behind the digital twin-inspired idea.

In [ ]:
v = tables["visits"]
traj = c2.merge(v, on="patient_id")
traj["weeks"] = (traj.visit_date - traj.t0).dt.days / 7
traj = traj[(traj.weeks >= -4) & ((traj.end_date.isna()) | (traj.visit_date <= traj.end_date)) & (traj.weeks <= 60)]
traj = traj.merge(resp[["patient_id", "episode_id", "pasi90"]], on=["patient_id", "episode_id"])

fig, ax = plt.subplots(figsize=(8, 4))
for (pid, ep), g in traj.groupby(["patient_id", "episode_id"]):
    color = {1.0: "tab:green", 0.0: "tab:red"}.get(g.pasi90.iloc[0], "lightgrey")
    ax.plot(g.weeks, g.pasi, color=color, alpha=.35, lw=1)
ax.set_xlabel("Weeks since treatment start"); ax.set_ylabel("PASI")
ax.set_title("PASI trajectories (green = PASI90 responder, red = non-responder, grey = not assessed)")
plt.tight_layout(); plt.savefig(path("figures") / "pasi_trajectories.png", dpi=200); plt.show()

## 5. Relationships between variables

In [ ]:
cols = ["pasi_baseline", "bsa_baseline", "dlqi_baseline", "age", "bmi", "disease_duration"]
corr = df2[cols].corr(method="spearman").round(2)
corr

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
assessed.boxplot(column="pasi_baseline", by="pasi90", ax=ax)
ax.set_title("Baseline PASI by PASI90 response"); plt.suptitle(""); ax.set_xlabel("PASI90 (0 = no, 1 = yes)")
plt.tight_layout(); plt.show()

## 6. Model 1 cohort (escalation)
Only meaningful with data that include patients starting on topical treatment (not available in the public dataset).

In [ ]:
from src.data.cohorts import model1_cohort
from src.outcomes.escalation import compute_escalation

c1 = model1_cohort(tables)
if len(c1):
    esc = compute_escalation(tables, c1)
    print(f"{len(c1)} patients starting on topical treatment, {int(esc.event.sum())} escalations")
    display(esc[["escalation_182d", "escalation_365d"]].agg(["mean", "count"]).round(2))
    display(esc.escalation_class.value_counts())
else:
    print("No patients starting on topical treatment in this source: Model 1 cannot be studied.")

## 7. Missing values in the modelling dataset

In [ ]:
(df2[num + binv].isna().mean() * 100).round(1).sort_values(ascending=False)